In [1]:
from astropy.io import fits
from astropy.table import Table, vstack, MaskedColumn
import numpy as np
import h5py
import math
import os
from tqdm import tqdm
import pandas as pd

In [ ]:
SGA = Table.read('')

In [ ]:
# Look at morph types in SGA
unique, counts = np.unique(SGA['Morphology'], return_counts = True)
print(unique, counts)

In [ ]:
if 'T_TYPE' in SGA.colnames:
    SGA.remove_column('T_TYPE')

In [ ]:
# Adds a new column named 'T_TYPE' to the 'SGA' dataset filled with NaN (Not a Number) values.
SGA.add_column(30, name='T_TYPE')

# assign numerical values to the 'T_TYPE' column based on the 'MORPHTYPE' values:
# Elliptical and dwarf ellipticals (MORPHTYPE == 'E'): Assigns a value of -5 to the 'T_TYPE' column for these types.
# 45468 
SGA['T_TYPE'][SGA['Morphology'] == 'E'] = -5

# Lenticulars (MORPHTYPE == 'S0' and 'S0-a'): Assigns values of -2 and 0, respectively.
# 4749
SGA['T_TYPE'][SGA['Morphology'] == 'S0'] = -2
#14527
SGA['T_TYPE'][SGA['Morphology'] == 'S0-a'] = 0

# Spirals (MORPHTYPE == 'Sa', 'Sab', 'Sb', 'Sbc', 'Sc', 'Scd', 'Sd'): Assigns values 1, 2, 3, 4, 5, 6, and 7, respectively.
# 2887 
SGA['T_TYPE'][SGA['Morphology'] == 'Sa'] = 1

# 5990
SGA['T_TYPE'][SGA['Morphology'] == 'Sab'] = 2

# 11076 
SGA['T_TYPE'][SGA['Morphology'] == 'Sb'] = 3

#17294 
SGA['T_TYPE'][SGA['Morphology'] == 'Sbc'] = 4

#28455 
SGA['T_TYPE'][SGA['Morphology'] == 'Sc'] = 5

#3382
SGA['T_TYPE'][SGA['Morphology'] == 'Scd'] = 6

#2470 
SGA['T_TYPE'][SGA['Morphology'] == 'Sd'] = 7

# Irregulars (MORPHTYPE == 'Sm', 'Irr', 'dIrr', 'I'): Assigns a value of 9 to 'Sm' and 10 to the others.
SGA['T_TYPE'][SGA['Morphology'] == 'Sm'] = 9
SGA['T_TYPE'][SGA['Morphology'] == 'Irr'] = 10
SGA['T_TYPE'][SGA['Morphology'] == 'dIrr'] = 10
SGA['T_TYPE'][SGA['Morphology'] == 'I'] = 10

# Dwarf spheroidal (MORPHTYPE == 'dSph'): Assigns a value of 11.
SGA['T_TYPE'][SGA['Morphology'] == 'dSph'] = 11

In [ ]:
# rearrange SGA table so that rows are ordered by the numerical values in the 'T_TYPE' column
SGA.sort('T_TYPE')

In [ ]:
# Ensure 'Main_Type' exists, initialized with 30
if 'Main_Type' in SGA.colnames:
    SGA.remove_column('Main_Type')
SGA['Main_Type'] = np.full(len(SGA), 30, dtype=int)

# Select only training galaxies
train_mask = SGA['Train-Test_Split'] == 'train'  # Assuming 'Split' column contains 'train' or 'test'

# Assign values only where train_mask is True
spirals = [1, 2, 3, 4, 5, 6, 7]
SGA['Main_Type'][train_mask & np.isin(SGA['T_TYPE'], spirals)] = 10  # Spirals
SGA['Main_Type'][train_mask & (SGA['T_TYPE'] == -5)] = 20  # Ellipticals
SGA['Main_Type'][train_mask & np.isin(SGA['T_TYPE'], [-2, 0])] = 0  # Lenticulars
SGA['Main_Type'][train_mask & (SGA['T_TYPE'] == 10)] = -5  # Irregulars

In [ ]:
# Group by ellipticals, spirals, lenticulars and irregulars

# Adds a new column named 'Main Type' to the SGA table and initializes it with 30s.
SGA.add_column(30, name='Main_Type') # undefined

spirals = [1, 2, 3, 4, 5, 6, 7]
for i in spirals:
    SGA['Main_Type'][SGA['T_TYPE'] == i] = 10 # spirals

SGA['Main_Type'][SGA['T_TYPE'] == -5] = 20 # ellipticals

SGA['Main_Type'][SGA['T_TYPE'] == -2] = 0 # len
SGA['Main_Type'][SGA['T_TYPE'] == 0] = 0 # len

#SGA['Main_Type'][SGA['T_TYPE'] == 9] = -5 # irregular
SGA['Main_Type'][SGA['T_TYPE'] == 10] = -5 # irregular

In [ ]:
# Ensure the column is of string type (if needed) and replace '--' with 30
SGA['Main_Type'] = [30 if str(val).strip() == '--' else val for val in SGA['Main_Type']]

# Print the updated table
print("\nUpdated SGA Table:")
print(SGA)

# Group by 'Main_Type' and count occurrences
counts = SGA.group_by('Main_Type')

# Create a summary table of counts
summary = Table({'Main_Type': counts.groups.keys, 'Count': [len(group) for group in counts.groups]})

# Print the summary of counts
print("\nCount of Each Main Type:")
print(summary)

In [ ]:
# identify and count the galaxies in the 'SGA' table that haven't been assigned specific types and are labeled as 'Undefined' in the 'Main_Type' column.
SGA[SGA['Main_Type'] == 30]
n = len(SGA[SGA['Main_Type'] == 30])

In [ ]:
# remove all rows with undefined morph type
SGA.sort('Main_Type')
SGA.remove_rows([range((len(SGA) - n),len(SGA))])
SGA

## Generating h5py file

In [ ]:
# Generating h5py file and generating datasets to store processed data

f = h5py.File('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/FPy3_it1.h5','a')

num_galaxies = 
count = 97465
pixel = 152
images = f.create_dataset('images',(count,3,pixel,pixel), maxshape=(None, 3, pixel, pixel))
ra = f.create_dataset('ra',(count,), maxshape=(None,))
dec = f.create_dataset('dec',(count,), maxshape=(None,))
mag_g = f.create_dataset('mag_g',(count,), maxshape=(None,))
mag_r = f.create_dataset('mag_r',(count,), maxshape=(None,))
mag_z = f.create_dataset('mag_z',(count,), maxshape=(None,))
cutout_type = f.create_dataset('cutout_type',(count,), maxshape=(None,))
umap_region = f.create_dataset('umap_region',(count,), maxshape=(None,))
morphology = f.create_dataset('morphology',(count,), maxshape=(None,))
Main_Type = f.create_dataset('Main_Type',(count,), maxshape=(None,))
SGA_ID = f.create_dataset('SGA_ID', (count,), maxshape=(None,))
targetid = f.create_dataset('targetid', (count,), maxshape=(None,))

In [ ]:
# Loop through each galaxy in the specified range
valid_entries = 0
i = 0

while valid_entries < num_galaxies and i < len(SGA):
    # pull path from SGA table
    path = SGA['Path'][i] # for 152x152
    
    if path != '--':
        if os.path.exists(path):
            try:
                with fits.open(path, ignore_missing_simple=True, ignore_missing_end=True) as fits_file:
                    img_data = fits_file[0].data

                    if img_data is None or img_data.shape != (3, 152, 152):
                        raise ValueError(f"Unexpected image shape or no data in {path}")

                    # Store the image and other data in the respective arrays
                    images[i] = img_data
                    ra[i] = SGA['RA'][i]
                    dec[i] = SGA['DEC'][i]
                    mag_g[i] = SGA['G_MAG_SB26'][i]
                    mag_r[i] = SGA['R_MAG_SB26'][i]
                    mag_z[i] = SGA['Z_MAG_SB26'][i]
                    cutout_type[i] = 1
                    Main_Type[i] = SGA['Main_Type'][i]
                    SGA_ID[i] = SGA['SGA_ID'][i]
                    
                    valid_entries += 1
                    if valid_entries % 200 == 0:
                        print('Processing', valid_entries)
            except Exception as e:
                print(f"Error processing file {path}: {e}")
    else:
        print(f"Invalid Path: {path}")
            
    i += 1

print(f"Finished processing {i} galaxies.")

In [ ]:
f = h5py.File('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/h5py-files/SGA2020_iteration2.h5','a').close

In [ ]:
# File path
file_path = '/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/h5py-files/SGA2020_iteration2.h5'

# Open HDF5 file
f = h5py.File(file_path, 'a')

# Number of galaxies to process
num_galaxies = 97465
count = 97465
pixel = 152

# Create datasets
images = f.create_dataset('images', (count, 3, pixel, pixel), maxshape=(None, 3, pixel, pixel))
ra = f.create_dataset('ra', (count,), maxshape=(None,))
dec = f.create_dataset('dec', (count,), maxshape=(None,))
mag_g = f.create_dataset('mag_g', (count,), maxshape=(None,))
mag_r = f.create_dataset('mag_r', (count,), maxshape=(None,))
mag_z = f.create_dataset('mag_z', (count,), maxshape=(None,))
cutout_type = f.create_dataset('cutout_type', (count,), maxshape=(None,))
umap_region = f.create_dataset('umap_region', (count,), maxshape=(None,))
morphology = f.create_dataset('morphology', (count,), maxshape=(None,))
Main_Type = f.create_dataset('Main_Type', (count,), maxshape=(None,))
SGA_ID = f.create_dataset('SGA_ID', (count,), maxshape=(None,))

# Loop through each galaxy in the specified range
valid_entries = 0
i = 0

# Initialize tqdm progress bar
with tqdm(total=num_galaxies, desc="Processing galaxies", unit="galaxy") as pbar:
    while valid_entries < num_galaxies and i < len(SGA):
        # Get the path from the SGA table
        path = SGA['Path'][i]  # for 152x152 images
        
        if path != '--' and os.path.exists(path):
            try:
                with fits.open(path, ignore_missing_simple=True, ignore_missing_end=True) as fits_file:
                    img_data = fits_file[0].data

                    if img_data is None or img_data.shape != (3, 152, 152):
                        raise ValueError(f"Unexpected image shape or no data in {path}")

                    # Store the image and other data in the respective arrays
                    images[i] = img_data
                    ra[i] = SGA['RA'][i]
                    dec[i] = SGA['DEC'][i]
                    mag_g[i] = SGA['G_MAG_SB26'][i]
                    mag_r[i] = SGA['R_MAG_SB26'][i]
                    mag_z[i] = SGA['Z_MAG_SB26'][i]
                    cutout_type[i] = 1
                    Main_Type[i] = SGA['Main_Type'][i]
                    SGA_ID[i] = SGA['SGA_ID'][i]

                    valid_entries += 1
                    pbar.update(1)  # Update progress bar for each valid entry

            except Exception as e:
                print(f"Error processing file {path}: {e}")

        i += 1

print(f"Finished processing {valid_entries} galaxies.")

# Close HDF5 file
f.close()